## 0. Set Memory Config (Run This First!)

In [ ]:
# This configuration optimizes PyTorch's CUDA memory allocation, preventing out-of-memory errors.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set")

PYTORCH_CUDA_ALLOC_CONF set


## 1. Install Dependencies

In [ ]:
!pip install -q pymupdf faiss-cpu sentence-transformers transformers accelerate bitsandbytes torch tqdm

## 2. Imports

In [ ]:
# This cell imports necessary libraries, sets up device (GPU/CPU) detection, and defines utility functions for GPU memory management.
import fitz
import re
import gc
import faiss
import torch
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def free_gpu_memory():
    """Call this between heavy operations to release cached GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        print(f"GPU: {allocated:.2f}GB allocated | {reserved:.2f}GB reserved")

print_gpu_memory()

Using device: cuda
GPU: 0.00GB allocated | 0.00GB reserved


## 3. PDF Parsing & Text Cleaning

In [ ]:
# This cell defines functions for cleaning text extracted from PDFs, extracting section headers, and parsing entire PDF documents into a list of text and metadata.
def clean_text(text):
    # Replace specific checkbox characters with text representations
    text = text.replace('☒', '[CHECKED]').replace('☐', '[UNCHECKED]')
    # # Remove 'Yes No' sequences that often appear in checkboxes without a selection
    # text = re.sub(r'\bYes\b\s*\bNo\b', '', text)
    # Remove document headers/footers specific to Apple and Tesla 10-K filings
    text = re.sub(r'Apple Inc\..*?Form 10-K \| \d+', '', text)
    text = re.sub(r'Tesla, Inc\..*?Form 10-K \| \d+', '', text)
    # Remove trademark, registered, and copyright symbols
    text = re.sub(r'[®™©]', '', text)
    # Remove hyphenation at line breaks
    text = re.sub(r'-\n', '', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_section(text):
    # Attempt to find common 10-K section headers like 'Item 1A.', 'Item 8', etc.
    match = re.search(r'(Item\s+\d+[A-Z]?\.?\s*[A-Za-z &,]+)', text)
    if match:
        # Return the first 50 characters of the matched section header
        return match.group().strip()[:50]
    return "Unknown Section"

def parse_pdf(file_path, doc_label):
    # Open the PDF document using fitz (PyMuPDF)
    doc = fitz.open(file_path)
    documents = []
    # Iterate through each page of the document
    for page_number in range(len(doc)):
        page = doc.load_page(page_number)
        # Extract and clean text from the current page
        text = clean_text(page.get_text())
        # Skip pages with no significant text after cleaning
        if not text:
            continue
        # Append the cleaned text and its metadata to the list of documents
        documents.append({
            "text": text,
            "metadata": {
                "document": doc_label,
                "section": extract_section(text),
                "page": page_number + 1
            }
        })
    return documents

# Parse the Apple and Tesla 10-K PDF documents
import gdown
import os

# ── Download PDFs from Google Drive ──
APPLE_URL = "https://drive.google.com/file/d/1_bGog38EdY-jbkeLlVkHCq2P7PKR0UPE/view?usp=sharing"
TESLA_URL = "https://drive.google.com/file/d/1vBPJD1jd-tvZSA95JRz95wIZdQT1wDhe/view?usp=sharing"

APPLE_PDF = "/content/apple_10k.pdf"
TESLA_PDF = "/content/tesla_10k.pdf"

if not os.path.exists(APPLE_PDF):
    print("Downloading Apple 10-K...")
    gdown.download(APPLE_URL, APPLE_PDF, quiet=False, fuzzy=True)
else:
    print("Apple PDF already exists")

if not os.path.exists(TESLA_PDF):
    print("Downloading Tesla 10-K...")
    gdown.download(TESLA_URL, TESLA_PDF, quiet=False, fuzzy=True)
else:
    print("Tesla PDF already exists")

# ── Parse PDFs using local paths ──
apple_docs = parse_pdf(APPLE_PDF, "Apple 10-K")
tesla_docs = parse_pdf(TESLA_PDF, "Tesla 10-K")
all_docs = apple_docs + tesla_docs
print(f"Total pages loaded: {len(all_docs)}")

Apple PDF already exists
Tesla PDF already exists
Total pages loaded: 251


## 4. Chunking

In [ ]:
def chunk_document(documents, chunk_size=400, overlap=80):
    chunks = []
    # Iterate through each document
    for doc in documents:
        text = doc['text']
        metadata = doc['metadata']
        start = 0
        chunk_index = 0
        # Slide a window across the document text to create chunks
        while start < len(text):
            # Create a copy of the original metadata and add chunk index
            chunk_meta = dict(metadata)
            chunk_meta['chunk_index'] = chunk_index
            # Append the current chunk of text and its metadata
            chunks.append({"text": text[start:start+chunk_size], 'metadata': chunk_meta})
            # Move the start position by (chunk_size - overlap)
            start += chunk_size - overlap
            chunk_index += 1
    return chunks

# Chunk all parsed documents with specified size and overlap
chunks = chunk_document(all_docs, chunk_size=400, overlap=80)
print(f"Total chunks: {len(chunks)}")

Total chunks: 2763


## 5. Embedding & FAISS Index

In [ ]:
# FIX: Force embedder to CPU — saves ~1.5GB GPU VRAM for Mistral
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="cpu")

print("Encoding all chunks on CPU (takes ~10-15 mins)...")
# Extract text from chunks for embedding
texts = [chunk["text"] for chunk in chunks]
# Encode the texts into embeddings
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=64)
# Convert embeddings to float32 for FAISS compatibility
embeddings = embeddings.astype("float32")
# Normalize embeddings to unit length (L2 normalization) for cosine similarity with IP index
faiss.normalize_L2(embeddings)

# Get the dimension of the embeddings
dimension = embeddings.shape[1]
# Create a FAISS index using Inner Product (IP) for similarity search
index = faiss.IndexFlatIP(dimension)
# Add the generated embeddings to the FAISS index
index.add(embeddings)

free_gpu_memory()
print(f"\n FAISS index built: {index.ntotal} vectors")
print_gpu_memory()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding all chunks on CPU (takes ~10-15 mins)...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]


 FAISS index built: 2763 vectors
GPU: 0.00GB allocated | 0.00GB reserved


## 6. Reranker
**Memory fix:** Reranker runs on **CPU** too.

In [ ]:
# FIX: Force reranker to CPU — saves ~1GB GPU VRAM
reranker = CrossEncoder("BAAI/bge-reranker-large", device="cpu")
print(" Reranker loaded on CPU")
print_gpu_memory()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Reranker loaded on CPU
GPU: 0.00GB allocated | 0.00GB reserved


## 7. Retrieval & Reranking Functions

In [ ]:
def detect_document(query):
    # Convert query to lowercase to detect document based on keywords
    q = query.lower()
    # Check for 'apple' to return 'Apple 10-K'
    if "apple" in q: return "Apple 10-K"
    # Check for 'tesla' to return 'Tesla 10-K'
    if "tesla" in q: return "Tesla 10-K"
    # If no specific document is detected, return None
    return None

def retrieve_top_k(query, k=15, document=None):
    # Encode the query into an embedding
    query_embedding = embedder.encode([query]).astype("float32")
    # Normalize the query embedding for cosine similarity
    faiss.normalize_L2(query_embedding)
    # Search the FAISS index for top k*3 (or k) nearest neighbors
    distances, indices = index.search(query_embedding, k * 3 if document else k)
    # Retrieve the actual chunks using the indices
    results = [chunks[i] for i in indices[0]]
    # If a specific document is requested, filter the results
    if document:
        results = [c for c in results if c["metadata"]["document"] == document]
    # Return the top k results after filtering
    return results[:k]

def rerank(query, retrieved_chunks):
    # If no chunks were retrieved, return an empty list
    if not retrieved_chunks:
        return []
    # Create pairs of (query, chunk_text) for the reranker
    pairs = [(query, chunk["text"]) for chunk in retrieved_chunks]
    # Predict relevance scores using the CrossEncoder reranker
    scores = reranker.predict(pairs)
    # Sort chunks by relevance score in descending order
    return [chunk for _, chunk in sorted(zip(scores, retrieved_chunks), key=lambda x: x[0], reverse=True)]

## 8. Load Mistral-7B-Instruct
**Memory fixes applied:**
- `load_in_4bit=True` — quantizes weights to 4-bit (~4GB)
- `bnb_4bit_use_double_quant=True` — extra compression
- `low_cpu_mem_usage=True` — avoids doubling RAM during load
- `device_map="auto"` — lets accelerate manage placement

In [ ]:
free_gpu_memory()
print_gpu_memory()

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Configure 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Enable 4-bit quantization
    bnb_4bit_use_double_quant=True,   # Apply nested quantization for further memory savings
    bnb_4bit_quant_type="nf4", # Use NF4 quantization type (NormalFloat 4-bit)
    bnb_4bit_compute_dtype=torch.float16 # Use float16 for computation during quantization
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Set padding token to EOS token
tokenizer.padding_side = "left" # Set padding to the left for generation tasks

print("Loading Mistral-7B in 4-bit quantization (~3 mins)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto", # Automatically map model layers to available devices (GPU/CPU)
    low_cpu_mem_usage=True,           # FIX: avoids RAM spike during load by loading weights directly to GPU
)
model.eval()

free_gpu_memory()
print("\n Mistral-7B-Instruct loaded!")
print_gpu_memory()

GPU: 0.00GB allocated | 0.00GB reserved
Loading tokenizer...
Loading Mistral-7B in 4-bit quantization (~3 mins)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


 Mistral-7B-Instruct loaded!
GPU: 4.13GB allocated | 4.48GB reserved


## 9. Prompt Builder — Mistral [INST] Format

In [ ]:
def build_prompt(query, context_chunks, max_chars_per_chunk=2000):
    context_text = ""
    # Iterate through each context chunk to build a formatted context string
    for chunk in context_chunks:
        context_text += (
            f"Document: {chunk['metadata']['document']}\n"
            f"Section: {chunk['metadata']['section']}\n"
            f"Page: {chunk['metadata']['page']}\n"
            f"Content: {chunk['text'][:max_chars_per_chunk]}\n\n"
        )

    # Construct the full prompt for the Mistral model
    prompt = (
        f"<s>[INST] You are a financial analyst assistant. "
        f"Use ONLY the context below to answer the question.\n\n"
        f"STRICT RULES:\n"
        f"1. If the answer IS in the context: give a short direct answer with the exact number/fact.\n"
        f"2. If the question asks about something NOT in the context (forecasts, opinions, physical descriptions, current roles): "
        f"reply exactly: 'This question cannot be answered based on the provided documents.'\n"
        f"3. If the topic is in scope but the specific detail is missing: "
        f"reply exactly: 'Not specified in the document.'\n"
        f"4. Never guess. Never use outside knowledge.\n\n"
        f"Context:\n{context_text}\n"
        f"Question: {query} [/INST]"
        f"Answer: "
    )
    return prompt

## 10. Answer Function
**Memory fix:** `max_length=2048` and `max_new_tokens=150` kept tight to avoid OOM during generation.

In [ ]:
def answer_question(query: str) -> dict:
    # Detect if the query specifies a particular document (Apple or Tesla)
    doc_name = detect_document(query)

    # Retrieve initial top K chunks based on the query, optionally filtered by document
    retrieved = retrieve_top_k(query, k=15, document=doc_name)
    # Rerank the retrieved chunks to get the most relevant ones
    reranked  = rerank(query, retrieved)
    # Select the very top chunks after reranking for context
    top_chunks = reranked[:5]

    # Build the prompt for the Mistral model using the query and top chunks
    prompt = build_prompt(query, top_chunks)
    # Free up GPU memory before generating the answer
    free_gpu_memory()

    # Tokenize the prompt and move it to the device (GPU if available)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    # Generate the answer using the Mistral model without tracking gradients
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Extract newly generated tokens (the answer part) from the output
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    # Decode the new tokens into human-readable text and strip whitespace
    answer_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Split the answer into sentences and take only the first one for conciseness
    sentences = re.split(r'(?<=[.!?])\s', answer_text)
    answer_text = sentences[0] if sentences else answer_text

    # Clean up tensors and free GPU memory again
    del inputs, output_ids, new_tokens
    free_gpu_memory()

    # Determine if the answer indicates refusal and format sources accordingly
    refused = "cannot be answered" in answer_text or "Not specified" in answer_text
    if refused:
        sources = []
    else:
        # Extract source metadata from the top-ranked chunk
        top = top_chunks[0]["metadata"]
        sources = [
            top["document"],
            top["section"],
            f"p. {top['page']}"
        ]

    return {"answer": answer_text, "sources": sources}

## 11. Run Questions

In [ ]:
import json

questions = [
    {"question_id": 1,  "question": "What was Apples total revenue for the fiscal year ended September 28, 2024?"},
    {"question_id": 2,  "question": "How many shares of common stock were issued and outstanding as of October 18, 2024?"},
    {"question_id": 3,  "question": "What is the total amount of term debt (current + non-current) reported by Apple as of September 28, 2024?"},
    {"question_id": 4,  "question": "On what date was Apples 10-K report for 2024 signed and filed with the SEC?"},
    {"question_id": 5,  "question": "Does Apple have any unresolved staff comments from the SEC as of this filing? How do you know?"},
    {"question_id": 6,  "question": "What was Teslas total revenue for the year ended December 31, 2023?"},
    {"question_id": 7,  "question": "What percentage of Teslas total revenue in 2023 came from Automotive Sales (excluding Leasing)?"},
    {"question_id": 8,  "question": "What is the primary reason Tesla states for being highly dependent on Elon Musk?"},
    {"question_id": 9,  "question": "What types of vehicles does Tesla currently produce and deliver?"},
    {"question_id": 10, "question": "What is the purpose of Teslas lease pass-through fund arrangements?"},
    {"question_id": 11, "question": "What is Teslas stock price forecast for 2025?"},
    {"question_id": 12, "question": "Who is the CFO of Apple as of 2025?"},
    {"question_id": 13, "question": "What color is Teslas headquarters painted?"},
]

results = []
for q in questions:
    print(f"Running Q{q['question_id']}...", end=" ")
    result = answer_question(q["question"])
    results.append({
        "question_id": q["question_id"],
        "answer": result["answer"],
        "sources": result["sources"]
    })
    print(f"{result['answer'][:150]}...")

# Pretty print final JSON
print("\n\n===== FINAL OUTPUT =====")
print(json.dumps(results, indent=2))

Running Q1... Apple's total revenue for the fiscal year ended September 28, 2024 was 391,035 million....
Running Q2... 15,115,823,000 shares were issued and outstanding as of October 18, 2024....
Running Q3... 97,341 (millions)

Explanation: The document on page 46 states that the total term debt principal was $97,341 million as of September 28, 2024....
Running Q4... November 1, 2024....
Running Q5... 1....
Running Q6... 82,419 (millions)

Explanation: The context provides Tesla's total revenues for the year ended December 31, 2023, as stated in the Consolidated Statem...
Running Q7... 81.5% (Calculation: 82,419 - 2,120 = 80,299, then 80,299 / 90,738 * 100%)...
Running Q8... Tesla states that it is highly dependent on Elon Musk due to his role as CEO and his involvement in various management activities....
Running Q9... Tesla currently produces and delivers Model S, Model X, Model 3, Model Y, and Tesla Semi trucks....
Running Q10... In a lease pass-through fund arrangement, an investo

## 12. Debug — Check GPU Memory & Retrieved Chunks

In [ ]:
# # Check current GPU state
# print_gpu_memory()

# # Inspect retrieved chunks for any query
# debug_query = "Tesla total revenue automotive sales 2023"
# retrieved = retrieve_top_k(debug_query, k=10, document="Tesla 10-K")
# reranked  = rerank(debug_query, retrieved)

# print(f"\nTop 5 chunks for: '{debug_query}'\n")
# for i, chunk in enumerate(reranked[:5]):
#     meta = chunk['metadata']
#     print(f"--- Chunk {i+1} | {meta['document']} | Page {meta['page']} ---")
#     print(chunk['text'])
#     print()